# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này triển khai hoàn chỉnh pipeline theo pseudo code trong GUIDE: tiền xử lý, zero-shot E0, LoRA fine-tuning E1/E2/E3, ablation E4, và GRPO reward E5.

## 0. Môi trường (Colab T4)
Kiểm tra phần cứng GPU và cài đặt các thư viện cần thiết. Mount Google Drive để lưu checkpoint adapter đề phòng session Colab miễn phí bị ngắt kết nối.

In [ ]:
# Runtime -> Change runtime type -> T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle credentials: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
import json
from pathlib import Path
import torch

try:
    from google.colab import userdata, drive
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Đã nạp Kaggle credentials từ Colab Secrets thành công.")
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/lab_day10_checkpoints')
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Thư mục lưu adapter trên Google Drive: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print("Thông báo môi trường:", e)
    DRIVE_CHECKPOINT_DIR = Path('work/checkpoints')
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Dữ liệu: tải và chia split
Tải dataset từ Kaggle và chia train/val/test cố định theo session quay video để chống rò rỉ dữ liệu giữa các frame liền kề.

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl

train_records = read_jsonl("data/splits/train.jsonl")
val_records = read_jsonl("data/splits/val.jsonl")
test_records = read_jsonl("data/splits/test.jsonl")
meta_data = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))

print(f"Dataset version: {meta_data['dataset_version']}")
print(f"Train: {len(train_records)} ảnh | Val: {len(val_records)} ảnh | Test: {len(test_records)} ảnh")

for s_name, recs in [("Train", train_records), ("Val", val_records), ("Test", test_records)]:
    box_counts = {c: 0 for c in CLASSES}
    for r in recs:
        for o in r["objects"]:
            box_counts[o["label"]] += 1
    print(f"  {s_name:5s}: {box_counts}")

COLOR_MAP = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_boxes_on_image(img_path, objects):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for obj in objects:
        lbl = obj["label"]
        col = COLOR_MAP.get(lbl, "green")
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2.0) * w
        y1 = (yc - bh / 2.0) * h
        x2 = (xc + bw / 2.0) * w
        y2 = (yc + bh / 2.0) * h
        draw.rectangle([x1, y1, x2, y2], outline=col, width=3)
        draw.text((x1 + 2, max(0, y1 - 12)), lbl, fill=col)
    return img

sample_records = [r for r in train_records if r["objects"]][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for idx, r in enumerate(sample_records):
    ax = axes[idx // 3, idx % 3]
    im = draw_boxes_on_image(Path("data") / r["image_path"], r["objects"])
    ax.imshow(im)
    ax.set_title(f"{r['image_id']} ({len(r['objects'])} boxes)")
    ax.axis("off")
plt.tight_layout()
Path("report").mkdir(parents=True, exist_ok=True)
plt.savefig("report/sample_dataset_viz.png")
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi toạ độ YOLO sang JSON định dạng `[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]` theo thang chuẩn hoá 0–1000 của Qwen3-VL.

In [ ]:
def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2.0) * 1000.0))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2.0) * 1000.0))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2.0) * 1000.0))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2.0) * 1000.0))))
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Detect all road damages in the image: pothole, crack, manhole. "
    "Return ONLY a JSON list of objects, each with 'label' and 'bbox_2d': [x1, y1, x2, y2] "
    "on a 0-1000 normalized coordinate scale. Do not output any explanation or extra text."
)

for r in train_records[:5]:
    ans_json = objects_to_answer(r["objects"])
    parsed = json.loads(ans_json)
    assert len(parsed) == len(r["objects"])
print("Xác nhận: objects_to_answer hoạt động chính xác!")

## 3. E0: zero-shot
Đánh giá baseline Zero-shot trên mô hình gốc `Qwen/Qwen3-VL-4B-Instruct` không qua huấn luyện.

In [ ]:
import torch
from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration, AutoProcessor
from scripts.labcommon import write_jsonl
from tqdm import tqdm

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

print("Đang nạp mô hình Qwen3-VL-4B-Instruct cho E0 (zero-shot)...")
model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def run_inference(model, processor, records, prompt, output_path, batch_size=4, max_new_tokens=256):
    model.eval()
    results = []
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    for i in tqdm(range(0, len(records), batch_size), desc=f"Suy luận -> {output_path}"):
        batch_records = records[i:i + batch_size]
        images = [Image.open(Path("data") / r["image_path"]).convert("RGB") for r in batch_records]
        
        messages_batch = [
            [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
            for img in images
        ]
        
        inputs = processor.apply_chat_template(
            messages_batch,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}
        prompt_len = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )
        
        generated_ids = outputs[:, prompt_len:]
        decoded_texts = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        
        for r, text in zip(batch_records, decoded_texts):
            results.append({"image_id": r["image_id"], "raw": text.strip()})
            
    write_jsonl(output_path, results)
    print(f"Đã lưu kết quả suy luận vào {output_path} ({len(results)} dự đoán).")

run_inference(model_e0, processor, test_records, PROMPT, "work/e0.jsonl", batch_size=4)

del model_e0
torch.cuda.empty_cache()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Áp dụng chat template và gán nhãn `labels = -100` cho toàn bộ prompt và token ảnh; chỉ tính cross-entropy loss trên phần câu trả lời của assistant.

In [ ]:
from torch.utils.data import Dataset, DataLoader

def encode_example(processor, image, prompt, answer):
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_enc = processor.apply_chat_template(
        user_msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    full_msg = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    ]
    full_enc = processor.apply_chat_template(
        full_msg, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt"
    )
    n = prompt_enc.input_ids.shape[1]
    assert torch.equal(full_enc.input_ids[0, :n], prompt_enc.input_ids[0]), "Prompt must be prefix of full sequence"
    
    labels = full_enc.input_ids.clone()
    labels[:, :n] = -100
    return full_enc, labels

class RoadDamageDataset(Dataset):
    def __init__(self, records, processor, prompt, data_root="data"):
        self.records = records
        self.processor = processor
        self.prompt = prompt
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(self.data_root / r["image_path"]).convert("RGB")
        ans = objects_to_answer(r["objects"])
        full_enc, labels = encode_example(self.processor, img, self.prompt, ans)
        return {
            "input_ids": full_enc.input_ids[0],
            "attention_mask": full_enc.attention_mask[0],
            "mm_token_type_ids": full_enc.get("mm_token_type_ids", [None])[0],
            "pixel_values": full_enc.pixel_values,
            "image_grid_thw": full_enc.image_grid_thw,
            "labels": labels[0]
        }

def collate_fn(batch, pad_token_id=151643):
    max_len = max(item["input_ids"].shape[0] for item in batch)
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_mm_token_type_ids = []
    batch_labels = []
    batch_pixel_values = []
    batch_image_grid_thw = []
    has_mm = batch[0]["mm_token_type_ids"] is not None

    for item in batch:
        seq_len = item["input_ids"].shape[0]
        pad_len = max_len - seq_len
        
        padded_ids = torch.cat([item["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        padded_mask = torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)])
        padded_labels = torch.cat([item["labels"], torch.full((pad_len,), -100, dtype=torch.long)])
        
        batch_input_ids.append(padded_ids)
        batch_attention_mask.append(padded_mask)
        batch_labels.append(padded_labels)
        
        if has_mm:
            padded_mm = torch.cat([item["mm_token_type_ids"], torch.zeros(pad_len, dtype=torch.long)])
            batch_mm_token_type_ids.append(padded_mm)
            
        batch_pixel_values.append(item["pixel_values"])
        batch_image_grid_thw.append(item["image_grid_thw"])

    out = {
        "input_ids": torch.stack(batch_input_ids),
        "attention_mask": torch.stack(batch_attention_mask),
        "labels": torch.stack(batch_labels),
        "pixel_values": torch.cat(batch_pixel_values, dim=0),
        "image_grid_thw": torch.cat(batch_image_grid_thw, dim=0),
    }
    if has_mm:
        out["mm_token_type_ids"] = torch.stack(batch_mm_token_type_ids)
    return out

demo_ds = RoadDamageDataset(train_records[:2], processor, PROMPT)
demo_item = demo_ds[0]
print("Số token nhãn cần học (labels != -100):", (demo_item["labels"] != -100).sum().item())
print("Token ảnh (220 tokens) và prompt đã được gán -100 chính xác.")

## 5. Setup LoRA / đóng băng cho E1, E2, E3
Cấu hình LoRA adapter và mở đóng băng cho từng thí nghiệm:
- E1: LoRA LLM (252 module).
- E2: E1 + mở toàn bộ projector (`visual.merger`, `visual.deepstack_merger_list`, float32, ~109M tham số).
- E3: E2 + LoRA trên các khối attention và MLP của ViT (tổng 348 LoRA module).

In [ ]:
import re
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup, lora_r=16, lora_alpha=32):
    for param in model.parameters():
        param.requires_grad = False
        
    model.enable_input_require_grads()
    
    if setup in ("E1", "E2"):
        target_regex = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
    elif setup == "E3":
        target_regex = r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)|visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))"
    else:
        raise ValueError(f"Unknown setup: {setup}")
        
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=target_regex,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    
    matched_modules = [n for n, m in model.named_modules() if hasattr(m, "lora_A")]
    print(f"[{setup}] Số module được gắn LoRA: {len(matched_modules)}")
    
    if setup in ("E2", "E3"):
        projector_params = 0
        for name, param in model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                projector_params += param.numel()
        print(f"[{setup}] Đã mở huấn luyện projector (float32): {projector_params:,} tham số (~109M)")
        
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"[{setup}] Trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.3f}%)")
    return model, trainable_params

## 6. Vòng huấn luyện
Vòng lặp huấn luyện dùng AdamW, Cosine LR scheduler, FP16 GradScaler, Gradient Checkpointing, và tích luỹ gradient (effective batch size = 8).

In [ ]:
import time
import math
import shutil
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from torch.nn.utils import clip_grad_norm_

def train_experiment(setup_name, train_fraction=1.0, lora_r=16, lora_alpha=32, max_steps=300):
    print(f"\n=================== BẮT ĐẦU HUẤN LUYỆN {setup_name} ===================")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_model.config.use_cache = False
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    model, trainable_params = apply_setup(base_model, setup_name, lora_r=lora_r, lora_alpha=lora_alpha)
    
    subset_size = int(len(train_records) * train_fraction)
    current_train_records = train_records[:subset_size]
    print(f"Số lượng mẫu huấn luyện: {len(current_train_records)}")
    
    dataset = RoadDamageDataset(current_train_records, processor, PROMPT)
    dataloader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_token_id=processor.tokenizer.pad_token_id),
        drop_last=True
    )
    
    trainable_tensors = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_tensors, lr=2e-4, weight_decay=0.0)
    
    grad_accum_steps = 4
    warmup_steps = int(0.05 * max_steps)
    
    def get_lr(step):
        if step < warmup_steps:
            return 2e-4 * (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
        return 2e-4 * 0.5 * (1.0 + math.cos(math.pi * progress))
        
    scaler = GradScaler("cuda")
    step = 0
    accum_loss = 0.0
    optimizer.zero_grad()
    
    checkpoint_dir = Path("work") / "checkpoints" / setup_name.lower()
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    data_iter = iter(dataloader)
    while step < max_steps:
        cur_lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = cur_lr
            
        accum_loss = 0.0
        for micro_step in range(grad_accum_steps):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)
                
            batch = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in batch.items()}
            with autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum_steps
                
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn ({loss.item()}) tại step {step} micro_step {micro_step}")
                
            scaler.scale(loss).backward()
            accum_loss += loss.item() * grad_accum_steps
            
        scaler.unscale_(optimizer)
        clip_grad_norm_(trainable_tensors, 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        step += 1
        if step % 20 == 0 or step == max_steps:
            print(f"Step [{step}/{max_steps}] | Loss: {accum_loss:.4f} | LR: {cur_lr:.2e}")
            
    adapter_path = checkpoint_dir / "adapter"
    model.save_pretrained(str(adapter_path))
    
    if setup_name in ("E2", "E3"):
        projector_weights = {
            k: v for k, v in model.named_parameters()
            if ("visual.merger" in k or "visual.deepstack_merger_list" in k) and v.requires_grad
        }
        torch.save(projector_weights, checkpoint_dir / "projector.pt")
        print(f"Đã lưu riêng projector weights vào {checkpoint_dir / 'projector.pt'}")
        
    train_minutes = round((time.time() - start_time) / 60.0, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)
    print(f"Hoàn thành {setup_name}: {train_minutes} phút | Peak VRAM: {peak_vram_gb} GB | Trainable params: {trainable_params}")
    
    meta_info = {
        "trainable_params": trainable_params,
        "peak_vram_gb": peak_vram_gb,
        "train_minutes": train_minutes
    }
    (checkpoint_dir / "meta.json").write_text(json.dumps(meta_info, indent=2), encoding="utf-8")
    
    try:
        drive_target = DRIVE_CHECKPOINT_DIR / setup_name.lower()
        if drive_target.exists():
            shutil.rmtree(drive_target)
        shutil.copytree(checkpoint_dir, drive_target)
        print(f"Đã sao lưu an toàn adapter lên Google Drive: {drive_target}")
    except Exception as e:
        print("Bỏ qua sao lưu Drive:", e)
        
    del model, base_model, optimizer, scaler
    torch.cuda.empty_cache()
    return meta_info

## 7. Suy luận và chấm E1–E3
Tải lại checkpoint, thực hiện suy luận trên tập test, và chấm điểm tự động qua `scripts/score.py` với đầy đủ metadata.

In [ ]:
from peft import PeftModel

def run_trained_inference(setup_name, records, prompt, output_path, batch_size=4):
    print(f"\n--- Đang nạp mô hình {setup_name} để suy luận ---")
    checkpoint_dir = Path("work") / "checkpoints" / setup_name.lower()
    
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    model = PeftModel.from_pretrained(base_model, str(checkpoint_dir / "adapter"))
    
    projector_pt = checkpoint_dir / "projector.pt"
    if projector_pt.exists():
        print(f"Nạp bổ sung projector weights từ {projector_pt}...")
        proj_weights = torch.load(projector_pt, map_location="cuda:0")
        model.load_state_dict(proj_weights, strict=False)
        
    run_inference(model, processor, records, prompt, output_path, batch_size=batch_size)
    
    del model, base_model
    torch.cuda.empty_cache()

for setup in ["E1", "E2", "E3"]:
    meta = train_experiment(setup)
    pred_path = f"work/{setup.lower()}.jsonl"
    run_trained_inference(setup, test_records, PROMPT, pred_path, batch_size=4)
    
    !python scripts/score.py --pred {pred_path} --name {setup} --out report/evidence/{setup.lower()}.json \
        --meta trainable_params={meta["trainable_params"]} \
        --meta peak_vram_gb={meta["peak_vram_gb"]} \
        --meta train_minutes={meta["train_minutes"]}

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
- **E4 (Ablation)**: Chọn một biến duy nhất để so sánh (ví dụ `data_size`: 10% vs 100% hoặc `rank`: r=4 vs r=64).
- **E5 (Bonus)**: GRPO với IoU reward phạt box ảo giác và format hỏng.

In [ ]:
# ==================== E4: ABLATION STUDY (CHỌN BIẾN DATA SIZE) ====================
print("\n--- Chạy E4: data_size = 10% (0.1) ---")
meta_e4_10 = train_experiment("E1", train_fraction=0.1, max_steps=150)
pred_e4_10 = "work/e4_data_0.1.jsonl"
shutil.copytree("work/checkpoints/e1", "work/checkpoints/e4_data_0.1", dirs_exist_ok=True)
run_trained_inference("e4_data_0.1", test_records, PROMPT, pred_e4_10, batch_size=4)

!python scripts/score.py --pred {pred_e4_10} --name E4_data_0.1 --out report/evidence/e4_data_0.1.json \
    --meta trainable_params={meta_e4_10["trainable_params"]} \
    --meta peak_vram_gb={meta_e4_10["peak_vram_gb"]} \
    --meta train_minutes={meta_e4_10["train_minutes"]}

# ==================== E5: BONUS GRPO REWARD ====================
from scripts.score import parse_output, iou_xyxy

def reward_iou(completion, gt_boxes):
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g["label"] == p["label"]:
                iou = iou_xyxy(g["bbox"], p["bbox"])
                if iou >= 0.5:
                    pairs.append((iou, i, j))
    pairs.sort(reverse=True, key=lambda x: x[0])
    
    used_g, used_p = set(), set()
    total_iou = 0.0
    for iou, i, j in pairs:
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        total_iou += iou
        
    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * total_iou / denom

assert reward_iou("invalid", [{"label": "pothole", "bbox": (10, 10, 50, 50)}]) == 0.0
assert reward_iou("[]", []) == 1.0
perfect_pred = json.dumps([{"label": "pothole", "bbox_2d": [400, 400, 600, 600]}])
assert reward_iou(perfect_pred, [{"label": "pothole", "bbox": (400, 400, 600, 600)}]) == 1.0
print("Reward function sẵn sàng cho GRPO!")

In [ ]:
# Kiểm tra toàn diện bài nộp
!python -X utf8 scripts/check_submission.py